In [1]:
import qi
import os
import sys
import time
import threading

from naoqi import ALProxy

# Bot's IP address and port
ip = '10.104.23.217'
port = 9559

In [2]:
file_manager = ALProxy("ALFileManager", ip, port)
shared_folder_name = file_manager.getUserSharedFolderPath()
shared_folder_name

'/home/nao/'

In [ ]:
!pip install paramiko
import paramiko
def transfer_file(remote_path="/home/nao/microphones/recording.wav", local_path="pepper-bot/recordings/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.get(remote_path, local_path)

    sftp.close()
    ssh.close()

def receive_file(local_path="pepper-bot/recordings/recording.wav", remote_path="/home/nao/microphones/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.put(local_path, remote_path)

    sftp.close()
    ssh.close()

In [33]:
def eye_color(status="ListenOn", rgb = None):
    leds = ALProxy("ALLeds", ip, port)

    if rgb is None:
        if status == "ListenOn":
            leds.fadeRGB("AllLeds", "Yellow", 1)
        elif status == "SpeechDetected":
            leds.fadeRGB("AllLeds", "Green", 8)
        elif status == "EndOfProcess":
            leds.fadeRGB("AllLeds", "White", 1)

    else:
        leds.fadeRGB("AllLeds", rgb, 1)


# Callback function for timer
def timer_callback(timer_cb):
    timer_cb = True

# Records Audio when speech is detected
def record_audio(timer=8, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    speech_recogniser = ALProxy("ALSpeechRecognition", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    timer_cb = False

    speech_recogniser.subscribe("speech_recognition")
    speech_recogniser.setLanguage("English")
    vocabulary = ["pepper", "hello"]
    speech_recogniser.setVocabulary(vocabulary, False)

    # Stop recording when speech stops being detected or timer is reached
    if timer is not None:
        threading.Timer(timer, timer_callback, timer_cb).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Print status
        if debug:
            print(status)

        # Start recording when speech is detected
        if status == "SpeechDetected":
            eye_color(status)
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb:
            break

    if timer_cb:
        return

    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Stop recording when speech stops being detected
        if status == "EndOfProcess":
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            eye_color(status)
            break

        # Stop recording when timer is reached
        if timer_cb:
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break


def record_audio_sd(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    timer_cb = False

    sound_detector.subscribe("sound_detector")
    sound_detector.setParameter("Sensitivity", 0.1)

    # Record audio when sound is detected
    if timer is not None:
        threading.Timer(timer, timer_callback).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("ALSoundDetection/SoundDetected")

        # Print status
        if debug:
            print(status)

        # Start recording when sound is detected
        if status == 1:
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb:
            break

    if timer_cb:
        return
    
    while True:
        time.sleep(2)
        status = memory.getData("ALSoundDetection/SoundDetected")

        # Stop recording when sound stops being detected
        if status == 0:
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break

        # Stop recording when timer is reached
        if timer_cb:
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break

In [ ]:
record_audio(timer=5, debug=True)
transfer_file()

# Sam, put your code here, also your functions here too

In [ ]:
# Record Video
def record_video(timer=10):
    video_recorder = ALProxy("ALVideoRecorder", ip, port)
    video_recorder.startRecording("/home/nao/recordings/cameras", "test_video")
    time.sleep(timer)
    videoInfo = video_recorder.stopRecording()

    print('Video was saved on the robot: ', videoInfo[1])
    print('Total number of frames: ', videoInfo[0])

('Video was saved on the robot: ', '/home/nao/recordings/cameras/test_video.avi')
('Total number of frames: ', 73)


**Method 2**

In [ ]:
!pip install pyftplib
import pyftplib
from pyftplib import FTPServer
from pyftplib.handlers import FTPHandler
from pyftplib.authorizers import DummyAuthorizer
import threading

def ftp_transfer_file():
    remote_path = "/home/nao/microphones/recording.wav"
    authorizer = DummyAuthorizer()
    authorizer.add_user("nao", "nao", remote_path, perm="elradfmw")
    handler = FTPHandler
    handler.authorizer = authorizer
    server = FTPServer((ip, 21), handler)
    server_thread = threading.Thread(target=server.serve_forever)
    server_thread.daemon = True
    server_thread.start()

# Tablet

In [23]:
tablet_service = ALProxy("ALTabletService", ip, port)
tablet_service.wakeUp()

# Put code for tablet web view here